# Qwen3-4B — Fine-tune on ViNumQA (Unsloth)

Fine-tune `unsloth/Qwen3-4B` on the ViNumQA (VLSP-2025 numerical reasoning) train split: given
`(pre_text, table, post_text, question)` + the system instruction, learn to output the gold FinQA-style
computation program directly (no reasoning trace / chain-of-thought — that's a separate future effort,
likely via distillation from a larger teacher model).

This is functionally the same task as `sft/vsf-fintech-vinumqa-sft-qwen3-4b.ipynb`, but built on
**Unsloth** (`FastLanguageModel` + `trl.SFTTrainer`) instead of a hand-rolled `transformers.Trainer` +
`peft` loop, following the structure of `Qwen3_(14B)_Reasoning_Conversational.ipynb` in this same folder.
Run on Kaggle (T4/P100 free tier).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
from huggingface_hub import login

# Optional: only needed if you plan to push the adapter/merged model to the Hub.
# On Kaggle: Add-ons > Secrets > add "HF_TOKEN", then attach it to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("No HF_TOKEN found -- skipping login (fine unless you want to push_to_hub).")

### Load base model + LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # ViNumQA contexts (pre_text + table + post_text) can be long

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data prep

Load the official 3-way ViNumQA split and reuse the exact same context formatting (pre_text / markdown
table / post_text) and system prompt as the 0-shot/1-shot/SFT notebooks, so results stay comparable.

In [ ]:
import pandas as pd
from tabulate import tabulate

train_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/train.json')
valid_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/valid.json')
test_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/test.json')
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep the raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

train_df = process_split(train_df)
valid_df = process_split(valid_df)
test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).

### Build the conversational dataset

In [ ]:
def build_conversation(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": str(row["program"]).strip()},
    ]


def make_text_dataset(df):
    conversations = [build_conversation(row) for _, row in df.iterrows()]
    # enable_thinking=False: Qwen3's chat template still emits an empty
    # <think>\n\n</think>\n\n pair after the assistant tag even when disabled,
    # but no reasoning content is generated/trained on.
    texts = tokenizer.apply_chat_template(conversations, tokenize=False, enable_thinking=False)
    return texts


train_texts = make_text_dataset(train_df)
valid_texts = make_text_dataset(valid_df)
print(len(train_texts), len(valid_texts))
print(train_texts[0][:1000])

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_texts}).shuffle(seed=3407)
valid_dataset = Dataset.from_dict({"text": valid_texts})
train_dataset, valid_dataset

<a name="Train"></a>
### Train the model

We mask the loss to only the assistant turn (`train_on_responses_only`) so the model isn't penalized for
"predicting" the system prompt / context / question -- standard practice, and important here since the
context block can be much longer than the program string we actually want it to learn to produce.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/qwen3-4b-vinumqa-sft"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 8,   # effective batch size = 16
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 20,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Train the model. To resume a run, set `trainer.train(resume_from_checkpoint = True)`.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference

In [ ]:
FastLanguageModel.for_inference(model)

_row = test_df.iloc[0]
_prompt = USER_MESSAGE_FRAME.format(
    pre_text=_row["pre_text"], table=_row["table"], post_text=_row["post_text"], question=_row["question"],
)
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": _prompt},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=256,
    temperature=0.7, top_p=0.8, top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)
print("\nGold program:", _row["program"], "| Gold answer:", _row["answer"])

<a name="Save"></a>
### Saving

Saves LoRA adapters locally. Uncomment the `push_to_hub` lines (and set `HF_USERNAME`) to also upload.
See the merged-16bit / GGUF cells below for other export options (same as the reference notebook).

In [ ]:
ADAPTER_DIR = "/kaggle/working/qwen3-4b-vinumqa-sft-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter to {ADAPTER_DIR}")

# HF_USERNAME = "your_hf_username"
# model.push_to_hub(f"{HF_USERNAME}/qwen3-4b-vinumqa-sft-adapter", token=HF_TOKEN)
# tokenizer.push_to_hub(f"{HF_USERNAME}/qwen3-4b-vinumqa-sft-adapter", token=HF_TOKEN)

In [ ]:
# Merge to 16bit / 4bit, or export GGUF for llama.cpp -- disabled by default.
if False:
    model.save_pretrained_merged("qwen3-4b-vinumqa-sft-16bit", tokenizer, save_method="merged_16bit")
if False:
    model.save_pretrained_merged("qwen3-4b-vinumqa-sft-4bit", tokenizer, save_method="merged_4bit")
if False:
    model.save_pretrained_gguf("qwen3-4b-vinumqa-sft", tokenizer, quantization_method="q4_k_m")

<a name="Eval"></a>
### PA / EA on the ViNumQA test set

Quick check of Program Accuracy (PA) / Execution Accuracy (EA) after fine-tuning, using the same parser
as the 0-shot/1-shot/SFT notebooks so numbers are comparable. Even with `enable_thinking=False`, Qwen3's
chat template still emits an empty `<think>\n\n</think>\n\n` pair after the assistant tag, so we strip
up to and including the `</think>` token (id `151668`) before treating the rest as the generated program.

In [ ]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            # Multiples are written "14.3x" in these tables. Unlike "%", the
            # suffix carries no scaling -- the gold answer for such a row is the
            # plain multiple.
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced.

    A cell that merely marks a missing period ("-", "NA", an em dash) is
    skipped: rows in this dataset routinely lack a year or two, and voiding the
    whole row over one gap loses reductions the gold answers depend on. A cell
    with real but unreadable content still voids the row, so genuine parse
    failures are not silently averaged away.
    """
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF'].

    Bracket-depth aware, so a row label like `ROE (%)` stays one token. The
    original split on every bracket, which shattered such labels and broke the
    four-tokens-per-step structure the rest of the protocol relies on.

    Raises ValueError if trailing, non-whitespace text remains once no further
    step can be parsed (e.g. a step missing its closing paren, which happens
    both in a handful of gold programs and -- more importantly -- in model
    generations cut off by a max_new_tokens limit). An earlier version of this
    tokenizer silently stopped and returned only the steps parsed so far,
    which let a truncated program like "subtract(100, 50), divide(#0, 5"
    (missing text and closing paren) score as a valid, complete one-step
    program instead of being rejected -- a false positive for exactly the kind
    of generation failure this evaluator needs to catch.
    """
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching ')' found) in program: '{original_program}'"
            )

        program.append(m.group(1) + "(")
        # Split arguments on depth-0 commas so brackets inside a label survive.
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: '{text[pos:]}' (from: '{original_program}')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output.

    Bracket matched, so an outer call is never silently discarded: the earlier
    regex could only match a bracket-free call, so `multiply(divide(a, b), 100)`
    was reduced to its inner `divide(a, b)` and a percentage rescaling scored as
    if it were the gold answer.
    """
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text



def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples.

    The original walked the token list by joining it and splitting on ")", which
    silently mis-splits any argument containing a bracket -- exactly the row
    labels this dataset uses, e.g. `EPS (VND)`. Grouping the tokens directly is
    equivalent for well-formed programs and correct for those.
    """
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2).

    Same protocol as the official implementation -- literals become symbols,
    table steps become opaque variables, and the two expressions are compared
    after `simplify`, so a differently-arranged but algebraically identical
    program still counts. A prediction may only use symbols that appear in gold,
    which is what stops it from introducing a constant of its own (the `100` of
    a percentage rescaling, say). Only the step-splitting differs: it groups
    tokens rather than splitting a joined string on ")".
    """
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item.

    A generated_program that fails to tokenize (e.g. cut off mid-generation,
    missing a closing paren) scores (0.0, 0.0) rather than raising -- this is
    expected input from a real model, not a bug to surface as an exception.
    gold_program is assumed well-formed and is not caught the same way, so a
    malformed *gold* label still raises loudly instead of silently scoring 0.
    """
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary).

    `table_col` must hold the raw table (list of rows); without it, programs
    naming a table row cannot execute and score 0 on EA.
    """
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }


In [ ]:
import gc
from tqdm import tqdm

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

for df_index in tqdm(test_df.index, desc="Generating program..."):
    row = test_df.loc[df_index]
    prompt = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"], post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=256, temperature=0.7, top_p=0.8, top_k=20, min_p=0)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        idx = len(output_ids) - output_ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        idx = 0

    output = tokenizer.decode(output_ids[idx:], skip_special_tokens=True)
    test_df.at[df_index, "generated_program"] = output.strip()

    gc.collect()
    torch.cuda.empty_cache()

print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} samples generated.")

In [ ]:
df_scored, summary = evaluate_dataframe(test_df, table_col="table_raw")
print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}